# Notebook \#4: Anomaly Detection
#### by Sebastian Einar Salas Røkholt

---


**Index**  
- [**1 - Introduction and Setup**](#1---introduction)  
  - [*1.1 Setup*](#11-setup)  
  - [*1.2 Load the Anomaly Detection Model*](#12-load-the-anomaly-detection-model)  
  - [*1.3 Data Preparation*](#13-data-preparation)  
- [**2 - Running Anomaly Detection**](#2---running-anomaly-detection)  
  - [*2.1 Classifying a Single Charging Session*](#21-classifying-a-single-charging-session)  
  - [*2.2 Calculating Anomaly Statistics*](#22-calculating-anomaly-statistics) 
- [**3 - Plotting Anomalies**](#3---plotting-anomalies)  
  - [*3.1 Interactive Anomaly Plot*](#31-interactive-anomaly-plot)

---

## 1 - Introduction and Setup
write an introduction here

### 1.1 Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
from torch.utils.data import DataLoader
from mt4xai.models import MultiHorizonLSTM
from mt4xai.data import split_data, fit_scalers_on_train, apply_scalers, build_loader, get_session_from_loader
from mt4xai.ad import compute_session_errors, percentile_threshold, percentile_of_threshold, classify_by_threshold
from mt4xai.plot import make_bundle_from_session, plot_full_session

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
CLEANED_DATA_PATH = os.path.join(PROJECT_ROOT, "Data", "etron55-charging-sessions.parquet")

FINAL_MODEL_FOLDER_PATH = os.path.join(PROJECT_ROOT, "Models/final")
FINAL_MODEL_PATH = os.path.join(FINAL_MODEL_FOLDER_PATH, "final_model.pth")

PCT_THRESHOLD = 95.0  # Percentile for anomaly detection threshold
T_MIN_EVAL = 1  # Only evaluate and plot predictions for t >= T_MIN_EVAL + 1

print("[env] CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("[env] Device:", torch.cuda.get_device_name(torch.cuda.current_device()))
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

%matplotlib inline

[env] CUDA available: True
[env] Device: NVIDIA GeForce RTX 4070 Laptop GPU


### 1.2 Load the Anomaly Detection Model

In [2]:
# Load the model
def load_final_model(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device)
    cfg = checkpoint["config"]
    model = MultiHorizonLSTM(
        input_size=len(checkpoint["input_features"]),
        hidden_size=int(cfg["hidden_dim"]),
        horizon=cfg["horizon"],
        num_targets=len(checkpoint["target_features"]),
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"]
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model, checkpoint

model, checkpoint = load_final_model(FINAL_MODEL_PATH, device=DEVICE)

input_features  = checkpoint["input_features"]
target_features = checkpoint["target_features"]
cfg             = checkpoint["config"]
HORIZON         = int(cfg["horizon"])
POWER_WEIGHT    = float(cfg.get("power_weight", 0.5))

# Indices of targets within the *input* feature vector (used for reconstruction)
idx_power_inp = input_features.index("power")
idx_soc_inp   = input_features.index("soc")


/tmp/ipykernel_13775/2667936666.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


### 1.3 Data Preparation
In this section, we will load, transform, and split the data in exactly the same way as we did previously in order to extract the test set for running anomaly detection tests.

In [3]:
# Loads the cleaned data (already wrangled)
df_cleaned = pd.read_parquet(CLEANED_DATA_PATH)  # TODO: Replace with raw data and apply wrangling pipeline
df = df_cleaned.copy()

# Split exactly like the modelling notebook (GroupShuffleSplit + RANDOM_SEED)
train_df, val_df, test_df = split_data(df, test_size=0.2, validation_size=0.1)

# Fit scalers on train and apply to val/test (same as modelling)
cols_to_scale = list(set(input_features) | set(target_features))
scalers = fit_scalers_on_train(train_df, cols_to_scale)
power_scaler, soc_scaler = scalers["power"], scalers["soc"]  # Extract for later use
train_s = apply_scalers(train_df, scalers)
val_s   = apply_scalers(val_df, scalers)
test_s  = apply_scalers(test_df, scalers)

# Builds the DataLoaders to iterate over charging sessions
val_loader  = build_loader(val_s,  input_features, target_features, HORIZON, batch_size=16, shuffle=False, num_workers=0)
test_loader = build_loader(test_s, input_features, target_features, HORIZON, batch_size=16, shuffle=False, num_workers=0)

## 2 - Running Anomaly Detection

### 2.1 Classifying a single charging session 

In [4]:
# Computes the error distribution on validation set
df_val_err = compute_session_errors(model, val_loader, DEVICE, input_features, target_features,
                                    power_scaler, soc_scaler, POWER_WEIGHT, idx_power_inp, idx_soc_inp, T_MIN_EVAL)

# Defines the error threshold for anomalies as the nth (PCT_THRESHOLD-th) percentile of validation error
error_thr = percentile_threshold(df_val_err["error"].values, pct_thr=PCT_THRESHOLD)

# Compute its error the same way
session = get_session_from_loader(test_loader, session_idx=0, batch_idx=0, session_id=None)
session_id = session[0][0]  # Extract session_id bacause we selected the session by index (not by id)
error_df = compute_session_errors(model, session, DEVICE, input_features, target_features,
                                power_scaler, soc_scaler, POWER_WEIGHT, idx_power_inp, idx_soc_inp, T_MIN_EVAL)
error = round(float(error_df["error"].iloc[0]), 4)
label = "ABNORMAL" if error > error_thr else "NORMAL"

print(f"""Anomaly detection on session {session_id[0]}:
      f"Error = {error}\nAD threshold = {error_thr} \n Classification = {label}""")


Anomaly detection on session 5128923:
      f"Error = 1.0066
AD threshold = 9.653 
 Classification = NORMAL


In [5]:
# Compute its error the same way
session = get_session_from_loader(test_loader, session_id=1691162)
session_id = session[0][0]  # Extract session_id bacause we selected the session by index (not by id)
error_df = compute_session_errors(model, session, DEVICE, input_features, target_features,
                                power_scaler, soc_scaler, POWER_WEIGHT, idx_power_inp, idx_soc_inp, T_MIN_EVAL)
error = round(float(error_df["error"].iloc[0]), 4)
label = "ABNORMAL" if error > error_thr else "NORMAL"

print(f"""Anomaly detection on session {session_id[0]}:
      f"Error = {error}\nAD threshold = {error_thr} \n Classification = {label}""")

Anomaly detection on session 1691162:
      f"Error = 0.2469
AD threshold = 9.653 
 Classification = NORMAL


### 2.2 Calculating Anomaly Statistics

In [6]:
# Computes prediction error for all charging sessions in the test set
df_test_errs = compute_session_errors(
    model, test_loader, DEVICE,
    input_features, target_features,
    power_scaler, soc_scaler, POWER_WEIGHT,
    idx_power_inp, idx_soc_inp, T_MIN_EVAL
)

# Classify sessions based on the error threshold calculated in section 2.1
df_test_cls = classify_by_threshold(df_test_errs, error_thr)

# Calculate some statistics about the test set
all_errs = df_test_errs["error"].to_numpy()
all_errs_sorted = np.sort(all_errs)
min_err = all_errs_sorted[0]
max_err = all_errs_sorted[-1]
min_session_len = int(df_test_errs["length"].min())
max_session_len = int(df_test_errs["length"].max())

# Show some statistics
print("Classification counts:\n", df_test_cls["label"].value_counts().to_string())
print("\nLowest-error (most normal) sessions:\n", df_test_cls.sort_values("error").head(3))  # Selects 3 lowest-error sessions
print("\nHighest-error (most abnormal) examples:\n", df_test_cls.sort_values("error", ascending=False).head(3))  # Selects 3 highest-error sessions
print(f"\nError range: [{min_err:.3f}, {max_err:.3f}]")
print(f"Session length range: [{min_session_len}, {max_session_len}] time steps")


Classification counts:
 label
normal      11571
abnormal      612

Lowest-error (most normal) sessions:
    charging_id     error  length   label
0     10303976  0.118043      10  normal
1     11935991  0.136369      10  normal
2      7080318  0.146011      12  normal

Highest-error (most abnormal) examples:
        charging_id      error  length     label
12182     11732342  67.532005      15  abnormal
12181      1836322  47.172943      15  abnormal
12180     11156542  38.277706      17  abnormal

Error range: [0.118, 67.532]
Session length range: [8, 60] time steps


## 3 - Plotting Anomalies

### 3.1 Interactive Anomaly Plot

In [7]:
# Use caching for faster repeated access to session data
BUNDLE_CACHE = {}
def get_bundle(session_id):
    key = (type(session_id).__name__, session_id)
    if key not in BUNDLE_CACHE:
        BUNDLE_CACHE[key] = make_bundle_from_session(
            model=model, df_scaled=test_s, sid=session_id, device=DEVICE,
            input_features=input_features, target_features=target_features, horizon=HORIZON,
            power_scaler=power_scaler, soc_scaler=soc_scaler,
            idx_power_inp=idx_power_inp, idx_soc_inp=idx_soc_inp,
            sort_by="minutes_elapsed"
        )
    return BUNDLE_CACHE[key]


# Defines interactive widgets
w_cls = widgets.ToggleButtons(
    options=[("Normal", "normal"), ("Abnormal", "abnormal")],
    value="abnormal", description="Show"
)
w_target = widgets.SelectMultiple(
    options=["power", "soc"], value=("power", "soc"),
    description="Targets", rows=2
)
w_idx = widgets.IntSlider(value=0, min=0, max=0, step=1, description="Sample idx", continuous_update=False)
w_len = widgets.IntRangeSlider(
    value=[min_session_len, max_session_len], min=min_session_len, max=max_session_len, step=1,
    description="Length ∈", continuous_update=False
)
w_err = widgets.FloatRangeSlider(
    value=[min_err, max_err], min=min_err, max=max_err,
    step=(max_err-min_err)/500.0, description="Error ∈",
    readout_format=".3f", continuous_update=False
)
w_thr = widgets.FloatSlider(
    value=error_thr, min=min_err, max=max_err,
    step=(max_err-min_err)/500.0, description="Threshold",
    readout_format=".3f", continuous_update=False
)
w_thr_pct = widgets.HTML(value=f"<b>{percentile_of_threshold(all_errs_sorted, w_thr.value):.1f}th</b> percentile")
out = widgets.Output()


def filtered_df(cls_label, thr, len_lo, len_hi, err_lo, err_hi):
    """Return filtered DataFrame based on UI settings."""
    df = df_test_errs[
        (df_test_errs["length"] >= len_lo) & (df_test_errs["length"] <= len_hi) &
        (df_test_errs["error"]  >= err_lo) & (df_test_errs["error"]  <= err_hi)
    ].copy()
    df["label"] = np.where(df["error"] > thr, "abnormal", "normal")
    df = df[df["label"] == cls_label]
    df = df.sort_values("error", ascending=(cls_label == "normal")).reset_index(drop=True)
    return df


def update_index_range(*_):
    """Update sample index slider range and trigger render if needed."""
    df = filtered_df(w_cls.value, w_thr.value, w_len.value[0], w_len.value[1], w_err.value[0], w_err.value[1])
    n = len(df)
    w_idx.max = max(0, n - 1)
    updated_idx = False
    if w_idx.value > w_idx.max:
        w_idx.value = w_idx.max
        updated_idx = True
    w_thr_pct.value = f"<b>{percentile_of_threshold(all_errs_sorted, w_thr.value):.1f}th</b> percentile"
    if not updated_idx:
        render_plots()

def render_plots(*_):
    """Render the selected session and targets."""
    df = filtered_df(w_cls.value, w_thr.value, w_len.value[0], w_len.value[1], w_err.value[0], w_err.value[1])
    n = len(df)
    with out:
        clear_output(wait=True)
        if n == 0:
            print("No sessions match the current filter.")
            print(f"Show: {w_cls.value} | Targets: {list(w_target.value)}")
            print(f"Threshold: {w_thr.value:.3f} ({percentile_of_threshold(all_errs_sorted, w_thr.value):.1f}th pct)")
            print(f"Length ∈ [{w_len.value[0]}, {w_len.value[1]}], Error ∈ [{w_err.value[0]:.3f}, {w_err.value[1]:.3f}]")
            return
        row = df.iloc[min(w_idx.value, n-1)]
        sid, err, lbl = row["charging_id"], float(row["error"]), row["label"]
        bundle = get_bundle(sid)
        for tgt in list(w_target.value):
            plot_full_session(bundle, power_scaler, soc_scaler,
                              idx_power_inp, idx_soc_inp, t_min_eval=T_MIN_EVAL,
                              target=tgt, error=err, threshold=w_thr.value, label=lbl)
        print(f"Filtered set has {n} sessions, use the 'Sample idx' slider to browse through them")


# Uses observers to update dependent widgets and re-render
for w in (w_cls, w_thr, w_len, w_err):
    w.observe(update_index_range, names="value")
for w in (w_idx, w_target):
    w.observe(render_plots, names="value")


# Defines the layout of the UI (sliders, buttons)
ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_cls, w_target]),
        widgets.VBox([w_idx, w_len]),
        widgets.VBox([widgets.HBox([w_err]), widgets.HBox([w_thr, w_thr_pct])])
    ]),
    out])

# Displays the UI and the initial rendering of the plot
update_index_range()
display(ui)